# 6 · EM: `_log_prior` and `fit`

**Trusts:** everything so far: `_init_cpts`, `_expected_counts`, `CPT.from_counts`.

EM alternates two steps until the objective stops improving:

- **E-step:** expected counts under the current CPTs (notebook 5)
- **M-step:** new CPTs from those counts, `from_counts` (notebook 4)

We build `fit` as: **one iteration by hand → the objective → the loop with its stopping rule → compare → what EM buys us.** The toy's eight rows are too few to learn from, so we sample data from the toy CPTs, where we know the true answer.

In [ ]:
%load_ext autoreload
%autoreload 2

import itertools

import numpy as np
import pandas as pd
from toy import (
    CARDS,
    CO,
    DATA,
    EVIDENCE,
    NO2,
    S1,
    S4,
    STRUCTURE,
    TABLES,
    H,
    T,
    cpts,
    network,
    same,
    toy_bn,
)

from cse250a.modules.core import CPT

net = network()
bn = toy_bn()   # DiscreteBN with the hand-picked CPTs, not fitted

import matplotlib.pyplot as plt
from toy import sample

from cse250a.modules.models import DiscreteBN

data = sample(600, seed=0, missing=0.3)   # 30% of reference and sensor cells blanked
data.head()

## Step 1: one iteration by hand

In [ ]:
model = DiscreteBN(net)
model.cpts = model._init_cpts(data)                       # start (notebook 4)

counts, loglik = model._expected_counts(data)             # E-step (notebook 5)
after_one = {n: CPT.from_counts(n, ps, counts[n], model.alpha).table   # M-step
             for n, ps in net.structure.items()}

theirs = DiscreteBN(net, max_iter=1).fit(data)            # fit stops after one M-step
same("one iteration = fit(max_iter=1)", after_one, {n: c.table for n, c in theirs.cpts.items()})

## Step 2: the objective, `_log_prior`

With α > 0 the M-step is a **MAP** estimate: α pseudo-counts per cell act like a Dirichlet prior on each CPT row. The quantity EM then never decreases is

$$\underbrace{\textstyle\sum_\text{rows} \log P(\text{observed})}_{\text{log-likelihood}} \;+\; \underbrace{\alpha \textstyle\sum_\text{every CPT cell} \log \theta}_{\text{log-prior (up to a constant)}}$$

The prior term rewards CPTs that keep every cell away from 0.

In [ ]:
by_hand = model.alpha * sum(np.log(c.table).sum() for c in model.cpts.values())
same("log-prior", by_hand, model._log_prior())
print(f"objective at the start: {loglik:.2f} + {by_hand:.2f} = {loglik + by_hand:.2f}")

## Step 3: the loop and its stopping rule

Each pass: E-step, **record** the objective, **stop** if the gain since last time is below `tol × |objective|`, else M-step. The objective is recorded *before* the M-step, so entry *i* is the objective of the CPTs after *i* M-steps.

In [ ]:
def em(data, max_iter=100, tol=1e-4, alpha=1.0):
    m = DiscreteBN(net, alpha=alpha)
    m.cpts = m._init_cpts(data)
    history = []
    for _ in range(max_iter):
        counts, loglik = m._expected_counts(data)
        history.append(loglik + m._log_prior())
        if len(history) >= 2 and history[-1] - history[-2] < tol * abs(history[-2]):
            break
        m.cpts = {n: CPT.from_counts(n, ps, counts[n], alpha) for n, ps in net.structure.items()}
    return m.cpts, history

mine_cpts, mine_hist = em(data)
fitted = DiscreteBN(net).fit(data)

same("objective history", mine_hist, fitted.result.history, atol=1e-9)
same("final CPTs", {n: c.table for n, c in mine_cpts.items()},
     {n: c.table for n, c in fitted.cpts.items()})
print(f"{fitted.result.n_iter} iterations, converged: {fitted.result.converged}, monotone: {fitted.result.is_monotone()}")

In [ ]:
h = fitted.result.history
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(h, marker="o")
ax.set_xlabel("iteration"); ax.set_ylabel("log-lik + log-prior")
ax.set_title("EM never goes down")
plt.show()

**Never decreasing** is a theorem for EM, not luck. A drop in this curve means a bug in the E- or M-step. That's what `result.is_monotone()` checks.

## Step 4: with nothing missing, EM is just counting

On complete data every row is "complete", so the E-step returns the complete counts, the first M-step gives the counting answer, and the next E-step finds nothing to improve. The loop stops after the second record.

In [ ]:
complete_data = sample(600, seed=0, missing=0.0)
m = DiscreteBN(net).fit(complete_data)
counted = {n: CPT.from_counts(n, ps, m._complete_counts(complete_data)[n], 1.0).table
           for n, ps in net.structure.items()}
same("complete data: EM = counting", {n: c.table for n, c in m.cpts.items()}, counted)
print("iterations:", m.result.n_iter)

## Step 5: what EM buys over dropping incomplete rows

Same 30%-missing data. Complete-case counting (the start, `_init_cpts`) throws away every row with a gap in the family. EM uses them. How far is each from the **true** CPTs, as the amount of data grows?

In [ ]:
def error(tables):
    return max(np.abs(tables[n] - TABLES[n]).max() for n in TABLES)

for n in (200, 1000, 5000):
    d = sample(n, seed=1, missing=0.3)
    m = DiscreteBN(net)
    start = {k: c.table for k, c in m._init_cpts(d).items()}
    end = {k: c.table for k, c in m.fit(d).cpts.items()}
    print(f"n={n:5}:  complete-case error {error(start):.3f}   EM error {error(end):.3f}")

**Reading this honestly:** EM helps, but only modestly here, and the gap closes as n grows. That's expected. `sample` blanks cells **completely at random**, so dropping incomplete rows throws away data but doesn't bias what's left. Complete-case counting is just noisier, and EM's gain is efficiency.

EM matters much more in two situations:
- **When gaps aren't random.** If the CO analyzer tends to fail on high-pollution days, the complete rows under-represent high CO, and complete-case counting is biased. Whether that happens in the real data is worth checking in EDA.
- **For latent nodes,** where complete-case counting can't learn anything at all (next step).

## Step 6: a latent node

The same `fit` learns a node that is **never** observed. Here an Event above both gases (Phase 3b). The toy data wasn't generated with an Event, so this only shows the machinery: EM stays monotone, and we get P(Event | evidence) for every hour.

Latent states have **no fixed meaning**: EM may call the "busy" state 0 or 1 depending on the seed. Check which state lines up with what you expect (here: high gases) before naming it, or anchor it with `init`.

In [ ]:
from cse250a.modules.core import Network

EVENT = "Event"
latent_net = Network(
    {EVENT: (H,), CO: (EVENT,), NO2: (EVENT,), S1: (CO, T), S4: (NO2, CO)},
    {**CARDS, EVENT: 2},
    latent=(EVENT,),
).validate(data.columns)

lm = DiscreteBN(latent_net, max_iter=50).fit(data)
print("monotone:", lm.result.is_monotone(), " iterations:", lm.result.n_iter)
print("P(CO high | Event):", lm.cpts[CO].table[:, 1].round(2), " ← which state is 'busy'?")

post = lm.predict_proba(data, EVENT, EVIDENCE)
print(post.groupby(data[H].to_numpy()).mean().round(2).rename_axis("hour"))

## Done

Every method of `DiscreteBN` has now been rebuilt from first principles and matched against the class:

| notebook | method | rests on |
|---|---|---|
| 2 | `_unnormalized` | the factorization |
| 3 | `_joint`, `posterior`, `log_likelihood`, `_predict_proba` | `_unnormalized` |
| 4 | `_complete_counts`, `_init_cpts` | counting, `from_counts` |
| 5 | `_expected_counts` | `_joint` + the axis bookkeeping |
| 6 | `_log_prior`, `fit` | E-step + M-step |